# Loan Prediction Competition

In this notebook, we apply ensemble methods such as XGBoost for binary loan approval classification, featuring systematic hyperparameter optimization and SHAP model interpretability.


## 1. Setup & Configuration


In [ ]:
import os
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Configuration constants — edit DATA_DIR to point to a different directory
DATA_DIR = '.'
RANDOM_STATE = 42
CV_FOLDS = 5


**Dependencies:** pandas, numpy, scikit-learn, xgboost, shap, matplotlib, seaborn

Install with: `pip install -r requirements.txt`


## 2. Data Loading & Exploration

Train and test data are loaded once here and reused in later sections — no duplicate loads.


In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test_no_label.csv'))
print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
train_df.head()


In [ ]:
train_df.describe()


In [ ]:
# =====================================
# Exploración de distribuciones
# =====================================
import matplotlib.pyplot as plt
import seaborn as sns

# Definimos listas de variables
categoricas = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
               'Credit_History', 'Property_Area', 'Loan_Status']  # target incluida para ver balance
numericas = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']

# -------------------------
# Categóricas: conteo
# -------------------------
for col in categoricas:
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, data=train_df, palette="pastel")
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Conteo")
    plt.xticks(rotation=45)
    plt.show()

# -------------------------
# Numéricas: histograma + boxplot
# -------------------------
for col in numericas:
    fig, axes = plt.subplots(1,2, figsize=(12,4))

    # Histograma
    sns.histplot(train_df[col].dropna(), kde=True, ax=axes[0], color="skyblue")
    axes[0].set_title(f"Histograma de {col}")

    # Boxplot para ver outliers
    sns.boxplot(x=train_df[col], ax=axes[1], color="lightgreen")
    axes[1].set_title(f"Boxplot de {col}")

    plt.tight_layout()
    plt.show()

In [ ]:
# Revisión de valores nulos
missing_values = train_df.isnull().sum()
missing_percent = (missing_values / len(train_df)) * 100

missing_summary = pd.DataFrame({
    'Valores Nulos': missing_values,
    'Porcentaje (%)': missing_percent
}).sort_values(by='Porcentaje (%)', ascending=False)

print("Resumen de valores nulos:")
display(missing_summary)

## 3. Preprocessing & Feature Engineering

The preprocessing pipeline handles missing values via intelligent imputation, encodes categorical features, applies log transforms to reduce skew in income and loan amount columns, and engineers domain-specific risk features.


In [ ]:
def preprocess_loan_data(df):
    """
    Preprocessing pipeline for loan prediction data.

    Handles missing values via intelligent imputation, encodes categorical
    features, applies log transforms to reduce skew, and engineers
    domain-specific risk features.

    Args:
        df: Raw loan DataFrame (train or test set)

    Returns:
        Processed DataFrame ready for model training
    """

    # Crear copia para no modificar original
    data = df.copy()

    # ===================
    # 1. IMPUTACIÓN INTELIGENTE
    # ===================

    # ✅ CLAVE: Manejo inteligente de Credit_History (esto dio el salto en accuracy)
    data['Credit_History_Missing'] = data['Credit_History'].isna().astype(int)
    data['Credit_History'].fillna(1.0, inplace=True)

    # Imputaciones simples
    data['Self_Employed'].fillna('No', inplace=True)
    data['Loan_Amount_Term'].fillna(360.0, inplace=True)
    data['Gender'].fillna('Male', inplace=True)
    data['Dependents'].fillna('0', inplace=True)
    data['Married'].fillna('No', inplace=True)
    data['Education'].fillna('Graduate', inplace=True)
    data['CoapplicantIncome'].fillna(0.0, inplace=True)

    # Imputación por grupos para ApplicantIncome
    if data['ApplicantIncome'].isnull().sum() > 0:
        median_by_group = data.groupby(['Education', 'Self_Employed'])['ApplicantIncome'].median()
        for idx in data[data['ApplicantIncome'].isnull()].index:
            education = data.loc[idx, 'Education']
            self_employed = data.loc[idx, 'Self_Employed']
            if (education, self_employed) in median_by_group.index:
                data.loc[idx, 'ApplicantIncome'] = median_by_group[education, self_employed]
            else:
                data.loc[idx, 'ApplicantIncome'] = data['ApplicantIncome'].median()

    # Imputación por grupos para LoanAmount
    if data['LoanAmount'].isnull().sum() > 0:
        median_by_group = data.groupby(['Education', 'Property_Area'])['LoanAmount'].median()
        for idx in data[data['LoanAmount'].isnull()].index:
            education = data.loc[idx, 'Education']
            property_area = data.loc[idx, 'Property_Area']
            if (education, property_area) in median_by_group.index:
                data.loc[idx, 'LoanAmount'] = median_by_group[education, property_area]
            else:
                data.loc[idx, 'LoanAmount'] = data['LoanAmount'].median()

    # ===================
    # Log transforms reduce right skew in income and loan amount features.
    # Normalized distributions improve XGBoost split quality.
    # 2. TRANSFORMACIONES LOGARÍTMICAS
    # ===================

    data['log_ApplicantIncome'] = np.log(data['ApplicantIncome'])
    data['log_CoapplicantIncome'] = np.log1p(data['CoapplicantIncome'])
    data['log_LoanAmount'] = np.log(data['LoanAmount'])

    # ===================
    # 3. FEATURE ENGINEERING
    # ===================

    # Variables base
    total_income = data['ApplicantIncome'] + data['CoapplicantIncome']

    # Features principales
    data['total_income'] = total_income
    data['loan_to_income_ratio'] = data['LoanAmount'] / (total_income / 12)
    data['coapplicant_contribution'] = data['CoapplicantIncome'] / total_income

    # Income per person
    dependents_numeric = data['Dependents'].replace('3+', '3').astype(int) + 1
    data['income_per_person'] = total_income / dependents_numeric

    # Ratios logarítmicos
    log_monthly_payment = data['log_LoanAmount'] - np.log(data['Loan_Amount_Term'])
    log_monthly_income = np.log(total_income) - np.log(12)
    data['log_monthly_payment_to_income_ratio'] = log_monthly_payment - log_monthly_income

    # Categorización
    data['income_category'] = pd.cut(total_income,
                                   bins=[0, 3000, 6000, 10000, np.inf],
                                   labels=['Low', 'Medium', 'High', 'Very_High'])

    data['loan_size_category'] = pd.cut(data['LoanAmount'],
                                      bins=[0, 100, 150, 200, np.inf],
                                      labels=['Small', 'Medium', 'Large', 'Very_Large'])

    # Perfiles de riesgo y estabilidad
    # Engineered risk features capture domain knowledge about loan creditworthiness:
    # - high_risk_profile: applicant has multiple simultaneous risk factors
    # - financial_stability: stable married income, no outstanding debt
    data['high_risk_profile'] = ((data['Credit_History'] == 0) |
                                (data['Self_Employed'] == 'Yes') |
                                (data['loan_to_income_ratio'] > 0.4)).astype(int)

    married_flag = (data['Married'] == 'Yes').astype(int)
    education_flag = (data['Education'] == 'Graduate').astype(int)
    data['financial_stability'] = (married_flag &
                                  education_flag &
                                  (data['Credit_History'] == 1.0)).astype(int)

    # ✅ FEATURE INTERACTIONS (clave para el 80%)
    data['credit_missing_interaction'] = data['Credit_History'] * data['Credit_History_Missing']
    data['married_coapplicant_synergy'] = married_flag * (data['CoapplicantIncome'] > 0).astype(int)

    education_flag = (data['Education'] == 'Graduate').astype(int)
    self_employed_flag = (data['Self_Employed'] == 'Yes').astype(int)
    data['education_employment_interaction'] = education_flag * self_employed_flag

    # Wealthy conservative borrower
    income_q75 = total_income.quantile(0.75)
    loan_q25 = data['LoanAmount'].quantile(0.25)
    high_income_flag = (total_income > income_q75)
    small_loan_flag = (data['LoanAmount'] < loan_q25)
    data['wealthy_conservative_borrower'] = (high_income_flag & small_loan_flag).astype(int)

    # Term-amount mismatch
    long_term = (data['Loan_Amount_Term'] >= 360)
    short_term = (data['Loan_Amount_Term'] < 360)
    large_loan = (data['LoanAmount'] > data['LoanAmount'].quantile(0.75))
    small_loan = (data['LoanAmount'] <= data['LoanAmount'].quantile(0.25))
    data['term_amount_mismatch'] = ((long_term & small_loan) | (short_term & large_loan)).astype(int)

    # Property-income context
    urban_flag = (data['Property_Area'] == 'Urban')
    rural_flag = (data['Property_Area'] == 'Rural')
    high_income_percentile = (total_income > total_income.quantile(0.8))
    data['urban_high_income'] = (urban_flag & high_income_percentile).astype(int)
    data['rural_high_income'] = (rural_flag & high_income_percentile).astype(int)

    # ===================
    # 4. ENCODING
    # ===================

    # Variables binarias
    data['Gender_Male'] = (data['Gender'] == 'Male').astype(int)
    data['Married_Yes'] = (data['Married'] == 'Yes').astype(int)
    data['Self_Employed_Yes'] = (data['Self_Employed'] == 'Yes').astype(int)
    data['Education_Graduate'] = (data['Education'] == 'Graduate').astype(int)

    # One-Hot Encoding
    dependents_dummies = pd.get_dummies(data['Dependents'], prefix='Dependents', drop_first=True)
    property_dummies = pd.get_dummies(data['Property_Area'], prefix='Property_Area', drop_first=True)
    income_dummies = pd.get_dummies(data['income_category'], prefix='Income', drop_first=True)
    loan_size_dummies = pd.get_dummies(data['loan_size_category'], prefix='LoanSize', drop_first=True)

    # Concatenar dummies
    data = pd.concat([data, dependents_dummies, property_dummies,
                     income_dummies, loan_size_dummies], axis=1)

    # Target encoding
    if 'Loan_Status' in data.columns:
        data['Loan_Status'] = (data['Loan_Status'] == 'Y').astype(int)

    # ===================
    # 5. LIMPIEZA FINAL
    # ===================

    # Eliminar variables categóricas originales
    categorical_to_drop = ['Gender', 'Married', 'Self_Employed', 'Education',
                          'Dependents', 'Property_Area', 'income_category', 'loan_size_category']

    # Eliminar transformaciones menos útiles
    numerical_to_drop = ['log_CoapplicantIncome']

    # Eliminar solo las que existen
    all_to_drop = [col for col in categorical_to_drop + numerical_to_drop if col in data.columns]
    data.drop(all_to_drop, axis=1, inplace=True)

    return data

In [ ]:
# Illustrative: show feature count before and after preprocessing.
# The actual training pipeline in Section 4 re-applies this to train_df.
_processed = preprocess_loan_data(train_df.copy())
print(f'Features before preprocessing: {train_df.shape[1]}')
print(f'Features after preprocessing: {_processed.shape[1]}')
